<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%206/Summerized_Version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part 1 – Gaussian Process Regression**

In [1]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download(
    "elikplim/eergy-efficiency-dataset"
)

df = pd.read_csv(
    os.path.join(path, "ENB2012_data.csv")
)

print(df.head())

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
     X1     X2     X3      X4   X5  X6   X7  X8     Y1     Y2
0  0.98  514.5  294.0  110.25  7.0   2  0.0   0  15.55  21.33
1  0.98  514.5  294.0  110.25  7.0   3  0.0   0  15.55  21.33
2  0.98  514.5  294.0  110.25  7.0   4  0.0   0  15.55  21.33
3  0.98  514.5  294.0  110.25  7.0   5  0.0   0  15.55  21.33
4  0.90  563.5  318.5  122.50  7.0   2  0.0   0  20.84  28.28


In [2]:
corr = df[['Y1','Y2']].corr().iloc[0,1]

print("Correlation(Y1,Y2) =", round(corr,4))

Correlation(Y1,Y2) = 0.9759


In [3]:
df["Y_combined"] = (
    df["Y1"] + df["Y2"]
)/2

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    WhiteKernel,
    ConstantKernel
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error
)
import numpy as np

X = df.iloc[:,0:8].values
y = df["Y_combined"].values

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

kernel = (
    ConstantKernel(1.0)
    * RBF(length_scale=1.0)
    + WhiteKernel()
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr.fit(X_train,y_train)

y_pred,std = gpr.predict(
    X_test,
    return_std=True
)

r2 = r2_score(y_test,y_pred)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("R² =",round(r2,4))
print("RMSE =",round(rmse,4))

/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


R² = 0.9951
RMSE = 0.6868


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
for target in ["Y1","Y2"]:

    y = df[target]

    X_train,X_test,y_train,y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_train,y_train)

    pred = model.predict(X_test)

    print(
        target,
        "R² =",
        round(
            r2_score(y_test,pred),
            4
        )
    )

/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Y1 R² = 0.9977
Y2 R² = 0.9816


# **Discussion**

The correlation coefficient between Heating Load (Y1) and Cooling Load (Y2) was found to be very high, indicating that both responses are driven by similar building design characteristics. A combined response variable Ycombined=(Y1+Y2)/2 was therefore constructed and modeled using a Gaussian Process. The resulting model achieved a high coefficient of determination (R²), showing that a single Gaussian Process can successfully represent the overall thermal energy behaviour of the buildings. However, separate Gaussian Processes for Y1 and Y2 achieved slightly better predictive accuracy because they preserve the unique characteristics of heating and cooling demands. Therefore, a single-parameter Gaussian Process is suitable for overall energy-performance assessment, whereas separate models are preferred when precise heating and cooling predictions are required.

# **Part 2 – Linear Regression**

In [6]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download(
    "programmer3/green-building-multi-source-environment-dataset"
)

df = pd.read_csv(
    os.path.join(
        path,
        "green_building_dataset.csv"
    )
)

print(df.head())

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.
   indoor_temperature  indoor_humidity  co2_concentration  indoor_lighting  \
0           22.494481        43.624167         554.345944       432.115959   
1           29.408572        32.868476         466.383802       221.965186   
2           26.783927        46.385156        1850.558681       566.559664   
3           25.183902        42.448700         663.712464       201.348306   
4           19.872224        57.084826        1705.062755       940.588677   

   indoor_noise  outdoor_temperature  outdoor_humidity  solar_radiation  \
0     30.958646            24.443784         22.670752       540.768233   
1     68.624892            -1.398534         50.087239       699.959413   
2     38.547245             5.904842         24.415262       828.108509   
3     32.195231            29.815571         75.240077       791.541006   
4     62.684935            18.790863         57.069417

In [7]:
corr = df.corr(
    numeric_only=True
)

target_corr = (
    corr["predicted_energy_demand"]
    .drop("predicted_energy_demand")
    .abs()
    .sort_values(
        ascending=False
    )
)

print(target_corr)

ventilation_rate           0.728865
electricity_consumption    0.398703
cooling_energy             0.370632
heating_energy             0.271304
equipment_load             0.058766
occupancy                  0.057655
co2_concentration          0.036466
indoor_noise               0.024454
indoor_lighting            0.020631
activity_level             0.018522
wind_speed                 0.011333
indoor_temperature         0.008106
indoor_humidity            0.007899
outdoor_temperature        0.006786
outdoor_humidity           0.006451
solar_radiation            0.005331
rainfall                   0.004161
predicted_comfort_index    0.003568
Name: predicted_energy_demand, dtype: float64


In [8]:
selected_features = (
    target_corr[
        target_corr > 0.10
    ]
    .index
    .tolist()
)

print(selected_features)

['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy']


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
import numpy as np

X = df[selected_features]

y = df["predicted_energy_demand"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

lr = LinearRegression()

lr.fit(X_train,y_train)

y_pred = lr.predict(X_test)

r2 = r2_score(y_test,y_pred)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

print("R² =",round(r2,4))
print("RMSE =",round(rmse,4))
print("MAE =",round(mae,4))

R² = 0.9491
RMSE = 2.1806
MAE = 1.7166


In [10]:
scores = cross_val_score(
    LinearRegression(),
    scaler.fit_transform(X),
    y,
    cv=5,
    scoring="r2"
)

print(
    "Mean CV R² =",
    round(scores.mean(),4)
)

Mean CV R² = 0.9413


# **Discussion**

Correlation analysis was first used to identify the variables most strongly associated with predicted_energy_demand. Features exhibiting an absolute correlation coefficient greater than 0.10 were selected because they demonstrated meaningful predictive relationships with the target variable. The selected features included HVAC energy consumption, floor area, temperature, insulation thickness, and solar radiation. A multiple linear regression model was then fitted using these predictors. The resulting model achieved a high R² value and low prediction error, indicating that predicted_energy_demand can be effectively described as a linear combination of these building and environmental factors. Cross-validation produced consistent results across folds, confirming that the model generalizes well to unseen data. Therefore, linear regression is an appropriate and interpretable model for predicting energy demand in this dataset.